# 01 – Exploratory Data Analysis

**Project:** A Multi-Agent Deep Reinforcement Learning Framework with CNN-Encoded Alternative Data for Real-Time Credit Decisioning in Emerging Markets

**Dataset:** German Credit Data (UCI / Statlog) + Synthetic Sequential Alternative Data

This notebook loads the classic German Credit dataset, engineers a thin-file proxy, generates synthetic transaction sequences (alternative data), and performs exploratory analysis relevant to the research questions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
%matplotlib inline

ROOT = Path("..")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Libraries loaded successfully.")

## 1. Load German Credit Dataset

In [ ]:
col_names = [
    'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
    'savings', 'employment', 'installment_rate', 'personal_status_sex',
    'other_debtors', 'residence_since', 'property', 'age',
    'other_installment', 'housing', 'existing_credits', 'job',
    'num_dependents', 'telephone', 'foreign_worker', 'target'
]

df = pd.read_csv(DATA_RAW / "german.data", sep=' ', header=None, names=col_names)
print(f"Shape: {df.shape}")
print(f"\nTarget distribution (1=Good, 2=Bad):")
print(df['target'].value_counts())
print(f"\nDefault (Bad) rate: {(df['target']==2).mean():.2%}")
df.head()

## 2. Basic Cleaning & Feature Engineering

In [ ]:
df['default'] = (df['target'] == 2).astype(int)

# Thin-file proxy (emerging-market style)
df['thin_file_flag'] = (
    ((df['checking_status'] == 'A14') | (df['savings'].isin(['A61', 'A65']))) &
    (df['employment'].isin(['A71', 'A72']))
).astype(int)

print(f"Thin-file rate: {df['thin_file_flag'].mean():.2%}")
print(f"Default rate among thin-file: {df.loc[df['thin_file_flag']==1, 'default'].mean():.2%}")
print(f"Default rate among thick-file: {df.loc[df['thin_file_flag']==0, 'default'].mean():.2%}")

In [ ]:
num_cols = ['duration', 'credit_amount', 'installment_rate', 'residence_since',
            'age', 'existing_credits', 'num_dependents']

cat_cols = ['checking_status', 'credit_history', 'purpose', 'savings',
            'employment', 'personal_status_sex', 'property', 'housing', 'job']

df_encoded = pd.get_dummies(df[cat_cols], drop_first=True)
df_model = pd.concat([df[num_cols + ['thin_file_flag', 'default']], df_encoded], axis=1)

print(f"Model-ready shape: {df_model.shape}")
df_model.head()

## 3. Generate Synthetic Sequential Alternative Data (for CNN)

We create 30-day x 8-channel transaction sequences correlated with the default label. This simulates mobile-money / digital-footprint alternative data common in emerging markets.

In [ ]:
np.random.seed(42)
N = len(df)
SEQ_LEN = 30
N_CHANNELS = 8

txn_seq = np.random.randn(N, SEQ_LEN, N_CHANNELS).astype(np.float32) * 0.4

for i in range(N):
    if df['default'].iloc[i] == 1:
        txn_seq[i, -10:, 0] -= np.linspace(0.3, 0.9, 10)
        txn_seq[i, -10:, 1] += np.linspace(0.2, 0.7, 10)
        txn_seq[i, :, 2] += 0.4
    else:
        txn_seq[i, -10:, 0] += np.linspace(0.1, 0.5, 10)
        txn_seq[i, :, 3] += 0.3

print(f"Transaction sequence shape: {txn_seq.shape}")
print(f"Mean abs value (Bad): {np.abs(txn_seq[df['default']==1]).mean():.3f}")
print(f"Mean abs value (Good): {np.abs(txn_seq[df['default']==0]).mean():.3f}")

## 4. Key Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df['default'].value_counts(normalize=True).plot(kind='bar', ax=axes[0,0], color=['#2ecc71', '#e74c3c'])
axes[0,0].set_title('Default Rate (0=Good, 1=Bad)')
axes[0,0].set_ylabel('Proportion')
axes[0,0].set_xticklabels(['Good', 'Bad'], rotation=0)

df.groupby('thin_file_flag')['default'].mean().plot(kind='bar', ax=axes[0,1], color='#3498db')
axes[0,1].set_title('Default Rate by Thin-File Status')
axes[0,1].set_xlabel('Thin-file flag')
axes[0,1].set_ylabel('Default Rate')
axes[0,1].set_xticklabels(['Thick-file', 'Thin-file'], rotation=0)

sns.boxplot(data=df, x='default', y='credit_amount', ax=axes[1,0])
axes[1,0].set_title('Credit Amount by Default Status')
axes[1,0].set_xticklabels(['Good', 'Bad'])

sns.histplot(data=df, x='age', hue='default', bins=20, ax=axes[1,1], kde=True)
axes[1,1].set_title('Age Distribution by Default Status')

plt.tight_layout()
plt.savefig(DATA_PROCESSED / "eda_overview.png", dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/eda_overview.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mean_good = txn_seq[df['default']==0].mean(axis=0)
mean_bad  = txn_seq[df['default']==1].mean(axis=0)

sns.heatmap(mean_good.T, ax=axes[0], cmap='RdYlGn', center=0)
axes[0].set_title('Avg Transaction Sequence – Good Credit')
axes[0].set_xlabel('Day')
axes[0].set_ylabel('Channel')

sns.heatmap(mean_bad.T, ax=axes[1], cmap='RdYlGn', center=0)
axes[1].set_title('Avg Transaction Sequence – Bad Credit')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Channel')

plt.tight_layout()
plt.savefig(DATA_PROCESSED / "sequence_heatmap.png", dpi=120, bbox_inches='tight')
plt.show()

## 5. Save Processed Data

In [ ]:
df_model.to_csv(DATA_PROCESSED / "german_credit_model.csv", index=False)
np.save(DATA_PROCESSED / "transaction_sequences.npy", txn_seq)
df.to_csv(DATA_PROCESSED / "german_credit_full.csv", index=False)

print("Saved files:")
print("  - data/processed/german_credit_model.csv")
print("  - data/processed/transaction_sequences.npy")
print("  - data/processed/german_credit_full.csv")
print("\nEDA complete. Proceed to 02_Baseline_Models.ipynb")